<a href="https://colab.research.google.com/github/mathriddle/ColabTurtlePlus/blob/main/examples_version2/files/svg_converter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Use this tool to convert an svg image so that it can be used as a turtle image in ColabTurtlePlus. The function center_svg() will center the image at (0,0), optionally reflect its y-axis to be compatible with the orientation of the y-axis in turtle graphics, and scale the image to a desired size. The svg code to be converted must include either width/height pixel values or a viewBox (in pixels) for the centering to work.

In [ ]:
!pip install svgpathtools

In [ ]:
import re
import xml.etree.ElementTree as ET
from svgpathtools import parse_path, Path, Line, CubicBezier, \
    QuadraticBezier, Arc

In [ ]:
def center_svg(svg_code, scale=1, reflect_y=True):
    """Center an SVG at (0, 0), optionally reflect its y-axis,
    and scale its coordinates and stroke widths.

    Args:
        svg_code: SVG source code as a string.
        scale: Positive scaling factor; default is 1.
        reflect_y: If True (default), reverse the y-axis.
                    If False, preserve its direction.

    Supports rect, circle, ellipse, line, polygon, polyline,
    and path elements, along with common transform attributes.

    If viewBox is absent, width and height must be numeric
    lengths, optionally ending in 'px'.

    Generated numeric values have at most three decimal places.
    """

    if not isinstance(scale, (int, float)) or isinstance(scale, bool):
        raise TypeError("scale must be a positive number")
    if not scale > 0 or not float(scale) < float("inf"):
        raise ValueError("scale must be a positive finite number")
    if not isinstance(reflect_y, bool):
        raise TypeError("reflect_y must be a boolean")

    root = ET.fromstring(svg_code)

    # Preserve the SVG namespace.
    if root.tag.startswith("{"):
        namespace = root.tag.split("}")[0][1:]
        ET.register_namespace("", namespace)

    # Match numeric values, including scientific notation.
    number_pattern = (
        r"[-+]?(?:\d*\.\d+|\d+\.?)(?:[eE][-+]?\d+)?"
    )

    def fmt(value):
        """Format a number to at most three decimal places."""
        value = round(float(value), 3)
        if value == 0:
            value = 0.0
        return f"{value:.3f}".rstrip("0").rstrip(".")

    # Determine the original viewBox.
    def parse_length(value):
        if value is None:
            raise ValueError(
                "SVG without viewBox must specify width and height"
            )

        match = re.fullmatch(
            r"\s*([-+]?(?:\d*\.\d+|\d+\.?)"
            r"(?:[eE][-+]?\d+)?)\s*(?:px)?\s*",
            value
        )
        if not match:
            raise ValueError(
                f"Cannot infer viewBox from {value!r}; "
                "use an explicit viewBox."
            )
        return float(match.group(1))

    vb = root.get("viewBox")

    if vb:
        values = re.split(r"[\s,]+", vb.strip())
        if len(values) != 4:
            raise ValueError("Invalid SVG viewBox")
        vx, vy, vw, vh = map(float, values)
    else:
        vx = vy = 0.0
        vw = parse_length(root.get("width"))
        vh = parse_length(root.get("height"))

    if vw <= 0 or vh <= 0:
        raise ValueError("viewBox width and height must be positive")

    cx = vx + vw / 2
    cy = vy + vh / 2

    # Transform individual coordinates.
    def sx(value):
        return fmt((float(value) - cx) * scale)

    def sy(value):
        y = cy - float(value) if reflect_y else float(value) - cy
        return fmt(y * scale)

    def sd(value):
        """Scale a length, radius, or displacement."""
        return fmt(float(value) * scale)

    def format_path_numbers(path_data):
        """Round all numeric values in generated path data."""
        return re.sub(
            number_pattern,
            lambda m: fmt(m.group()),
            path_data
        )

    # Transform polygon and polyline points.
    def shift_points(value):
        nums = [
            float(n) for n in re.findall(number_pattern, value)
        ]

        if len(nums) % 2:
            raise ValueError("Invalid polygon/polyline points")

        return " ".join(
            f"{sx(nums[i])},{sy(nums[i + 1])}"
            for i in range(0, len(nums), 2)
        )

    # Transform paths, including control points and elliptical arcs.
    def shift_path(d):
        path = parse_path(d)
        segments = []

        def point(z):
            return complex(
                (z.real - cx) * scale,
                (
                    cy - z.imag if reflect_y
                    else z.imag - cy
                ) * scale
            )

        for seg in path:
            if isinstance(seg, Line):
                segments.append(
                    Line(point(seg.start), point(seg.end))
                )

            elif isinstance(seg, CubicBezier):
                segments.append(
                    CubicBezier(
                        point(seg.start),
                        point(seg.control1),
                        point(seg.control2),
                        point(seg.end)
                    )
                )

            elif isinstance(seg, QuadraticBezier):
                segments.append(
                    QuadraticBezier(
                        point(seg.start),
                        point(seg.control),
                        point(seg.end)
                    )
                )

            elif isinstance(seg, Arc):
                radius = complex(
                    abs(seg.radius.real) * scale,
                    abs(seg.radius.imag) * scale
                )
                segments.append(
                    Arc(
                        point(seg.start),
                        radius,
                        -seg.rotation if reflect_y else seg.rotation,
                        seg.large_arc,
                        not seg.sweep if reflect_y else seg.sweep,
                        point(seg.end)
                    )
                )

            else:
                raise TypeError(
                    f"Unsupported path segment: {type(seg).__name__}"
                )

        return format_path_numbers(Path(*segments).d())

    # Transform common SVG transform attributes.
    def shift_transform(transform):
        pattern = re.compile(r"([A-Za-z]+)\s*\(([^)]*)\)")

        def replace(match):
            name = match.group(1)
            args = [
                float(v) for v in re.findall(
                    number_pattern, match.group(2)
                )
            ]

            if name == "translate":
                if len(args) == 1:
                    return f"translate({sd(args[0])})"
                if len(args) == 2:
                    ty = -args[1] if reflect_y else args[1]
                    return f"translate({sd(args[0])},{sd(ty)})"

            elif name == "rotate":
                angle = -args[0] if reflect_y else args[0]

                if len(args) == 1:
                    return f"rotate({fmt(angle)})"

                if len(args) == 3:
                    return (
                        f"rotate({fmt(angle)},"
                        f"{sx(args[1])},{sy(args[2])})"
                    )

            elif name in ("skewX", "skewY"):
                if len(args) == 1:
                    angle = -args[0] if reflect_y else args[0]
                    return f"{name}({fmt(angle)})"

            elif name == "scale":
                if len(args) in (1, 2):
                    return "scale(" + ",".join(
                        fmt(v) for v in args
                    ) + ")"

            elif name == "matrix":
                if len(args) == 6:
                    a, b, c, d, e, f = args

                    if reflect_y:
                        b, c, f = -b, -c, -f

                    return (
                        f"matrix({fmt(a)},{fmt(b)},"
                        f"{fmt(c)},{fmt(d)},"
                        f"{sx(e + cx)},{sy(f + cy)})"
                    )

            # Preserve transformations not explicitly handled.
            return match.group(0)

        return pattern.sub(replace, transform)

    # Process SVG elements.
    for elem in root.iter():
        tag = elem.tag.split("}")[-1]

        if tag == "rect":
            if "x" in elem.attrib:
                elem.set("x", sx(elem.get("x")))

            if "y" in elem.attrib:
                y = float(elem.get("y"))
                h = float(elem.get("height"))

                if reflect_y:
                    new_y = (cy - y - h) * scale
                else:
                    new_y = (y - cy) * scale

                elem.set("y", fmt(new_y))

            for attr in ("width", "height", "rx", "ry"):
                if attr in elem.attrib:
                    elem.set(attr, sd(elem.get(attr)))

        elif tag == "circle":
            if "cx" in elem.attrib:
                elem.set("cx", sx(elem.get("cx")))
            if "cy" in elem.attrib:
                elem.set("cy", sy(elem.get("cy")))
            if "r" in elem.attrib:
                elem.set("r", sd(elem.get("r")))

        elif tag == "ellipse":
            if "cx" in elem.attrib:
                elem.set("cx", sx(elem.get("cx")))
            if "cy" in elem.attrib:
                elem.set("cy", sy(elem.get("cy")))
            for attr in ("rx", "ry"):
                if attr in elem.attrib:
                    elem.set(attr, sd(elem.get(attr)))

        elif tag == "line":
            for attr in ("x1", "x2"):
                if attr in elem.attrib:
                    elem.set(attr, sx(elem.get(attr)))
            for attr in ("y1", "y2"):
                if attr in elem.attrib:
                    elem.set(attr, sy(elem.get(attr)))

        elif tag in ("polygon", "polyline"):
            if "points" in elem.attrib:
                elem.set("points", shift_points(elem.get("points")))

        elif tag == "path":
            if "d" in elem.attrib:
                elem.set("d", shift_path(elem.get("d")))

        # Scale stroke widths in presentation attributes.
        if "stroke-width" in elem.attrib:
            value = elem.get("stroke-width")
            match = re.fullmatch(
                r"\s*(" + number_pattern + r")\s*(.*)",
                value
            )
            if match:
                elem.set(
                    "stroke-width",
                    sd(match.group(1)) + match.group(2)
                )

        # Scale stroke widths in inline styles.
        if "style" in elem.attrib:
            def scale_stroke_width(match):
                value = float(match.group(2))
                unit = match.group(3) or ""
                return f"{match.group(1)}{sd(value)}{unit}"

            elem.set(
                "style",
                re.sub(
                    r"(stroke-width\s*:\s*)"
                    r"(" + number_pattern + r")"
                    r"([a-zA-Z%]*)",
                    scale_stroke_width,
                    elem.get("style")
                )
            )

        if "transform" in elem.attrib:
            elem.set(
                "transform",
                shift_transform(elem.get("transform"))
            )

    # Center the viewBox and scale its coordinate extent.
    root.set(
        "viewBox",
        f"{fmt(-vw * scale / 2)} "
        f"{fmt(-vh * scale / 2)} "
        f"{fmt(vw * scale)} {fmt(vh * scale)}"
    )

    return ET.tostring(root, encoding="unicode")

In [ ]:
star = """
 <svg xmlns="http://www.w3.org/2000/svg"
      width="304" height="290">
   <path d="M2,111 h300 l-242.7,176.3 92.7,-285.3 92.7,285.3 z"
      style="fill:#FB2;stroke:#BBB;stroke-width:15;stroke-linejoin:round"/>
</svg>
"""
result = center_svg(star, scale=0.25)
print(result)